In [20]:
%pip install meteostat

  Using cached meteostat-1.7.6-py3-none-any.whl.metadata (4.6 kB)
Using cached meteostat-1.7.6-py3-none-any.whl (33 kB)
Note: you may need to restart the kernel to use updated packages.


In [10]:
import requests
import pandas as pd
import json
import os
from dotenv import load_dotenv

# Load environment variables (Security)
load_dotenv()

OWM_API_KEY = os.getenv('OWM_API_KEY')
AVIATION_API_KEY = os.getenv('AVIATION_API_KEY')

# Target airports for the ETL pipeline
AIRPORTS = ['EPWA', 'EGLL', 'EDDF']

print("Libraries loaded. API keys secured and ready.")

Libraries loaded. API keys secured and ready.


In [22]:
# ==========================================
# PHASE: EXTRACT (Multiple Airports + Local Caching)
# ==========================================

print("Starting aviation data extraction...\n")

for airport in AIRPORTS:
    local_file = f"raw_flights_{airport}.json"

    # 1. CACHE CHECK: Avoid hitting API limits during development
    if os.path.exists(local_file):
        print(f"✅ Cache found: {local_file}. Skipping API call to preserve limits.")

    # 2. API REQUEST: Fetch only if cache doesn't exist
    else:
        print(f"--- Fetching data for {airport} from AviationStack API ---")
        url = f"http://api.aviationstack.com/v1/flights?access_key={AVIATION_API_KEY}&arr_icao={airport}"
        response = requests.get(url)

        if response.status_code == 200:
            json_data = response.json()
            with open(local_file, 'w', encoding='utf-8') as file:
                json.dump(json_data, file, ensure_ascii=False, indent=4)
            print(f"✅ Success! Data saved to {local_file}")
        else:
            print(f"❌ Error fetching {airport}: {response.status_code} - {response.text}")

print("\nExtract Phase (Aviation) completed!")

Starting aviation data extraction...

✅ Cache found: raw_flights_EPWA.json. Skipping API call to preserve limits.
✅ Cache found: raw_flights_EGLL.json. Skipping API call to preserve limits.
✅ Cache found: raw_flights_EDDF.json. Skipping API call to preserve limits.

Extract Phase (Aviation) completed!


In [12]:
# ==========================================
# PHASE: TRANSFORM (Data Cleaning & Normalization)
# ==========================================

import pandas as pd
import json

print("Starting data transformation in Pandas...\n")
dataframes_list = []
AIRPORTS = ['EPWA', 'EGLL', 'EDDF']

for airport in AIRPORTS:
    local_file = f"raw_flights_{airport}.json"
    with open(local_file, 'r', encoding='utf-8') as file:
        json_data = json.load(file)

    flights_list = json_data.get('data', [])

    temp_df = pd.json_normalize(flights_list)
    temp_df = temp_df.dropna(axis=1, how='all')
    dataframes_list.append(temp_df)

raw_flights_df = pd.concat(dataframes_list, ignore_index=True)

column_mapping = {
    'flight_date': 'flight_date',
    'flight_status': 'flight_status',
    'flight.iata': 'flight_number',
    'airline.name': 'airline',
    'departure.iata': 'departure_airport',
    'arrival.iata': 'arrival_airport',
    'arrival.scheduled': 'scheduled_arrival',
    'arrival.actual': 'actual_arrival',
    'arrival.delay': 'delay_minutes'
}

existing_columns = [col for col in column_mapping.keys() if col in raw_flights_df.columns]
clean_flights_df = raw_flights_df[existing_columns].rename(columns=column_mapping)

# --- BULLETPROOF DATA CLEANING LOGIC ---

# 1. Filter only flights that have already landed (only if 'flight_status' exists)
if 'flight_status' in clean_flights_df.columns:
    landed_flights = clean_flights_df[clean_flights_df['flight_status'] == 'landed'].copy()
else:
    landed_flights = clean_flights_df.copy()

# 2. Fill missing delay values with 0 safely
if 'delay_minutes' in landed_flights.columns:
    landed_flights['delay_minutes'] = landed_flights['delay_minutes'].fillna(0).astype(int)
else:
    landed_flights['delay_minutes'] = 0

# 3. Standardize datetime formats safely
if 'scheduled_arrival' in landed_flights.columns:
    landed_flights['scheduled_arrival'] = pd.to_datetime(landed_flights['scheduled_arrival'])

if 'actual_arrival' in landed_flights.columns:
    landed_flights['actual_arrival'] = pd.to_datetime(landed_flights['actual_arrival'])
    # Remove corrupted records without actual arrival time
    landed_flights.dropna(subset=['actual_arrival'], inplace=True)

print(f"✅ Transformation complete! {len(landed_flights)} landed flights ready for analysis.")

# Safe print for existing columns only
cols_to_print = [c for c in ['flight_number', 'airline', 'arrival_airport', 'delay_minutes'] if c in landed_flights.columns]
print(landed_flights[cols_to_print].head().to_string())

Starting data transformation in Pandas...

✅ Transformation complete! 0 landed flights ready for analysis.
Empty DataFrame
Columns: [flight_number, airline, arrival_airport, delay_minutes]
Index: []


In [21]:
# ==========================================
# PHASE: WEATHER EXTRACT (Meteostat Historical) & LOAD TO CSV
# ==========================================

from datetime import datetime, timedelta
from meteostat import Point, Hourly
import pandas as pd

print("\nStarting historical weather data extraction...\n")

# Meteostat uses exact geographical coordinates (Latitude, Longitude, Altitude)
# Here are the coordinates for our 3 target airports
airport_locations = {
    'EPWA': Point(52.1657, 20.9671, 110),  # Warsaw Chopin
    'EGLL': Point(51.4700, -0.4543, 25),   # London Heathrow
    'EDDF': Point(50.0333, 8.5705, 111)    # Frankfurt
}

# Define our time window: Last 24 hours
end_time = datetime.now()
start_time = end_time - timedelta(days=1)

weather_frames = []

for airport_code, location in airport_locations.items():
    print(f"Fetching hourly weather data for {airport_code}...")

    # Extract Hourly data from Meteostat
    data = Hourly(location, start_time, end_time)
    data = data.fetch()

    if not data.empty:
        # Meteostat returns data with 'time' as the index. We need it as a column.
        df_airport = data.reset_index()
        # Add our airport code so we can JOIN it later with flights
        df_airport['airport_code'] = airport_code
        weather_frames.append(df_airport)
    else:
        print(f"⚠️ WARNING: No weather data found for {airport_code} in the given timeframe.")

if weather_frames:
    # Combine all airports into one solid DataFrame
    weather_df = pd.concat(weather_frames, ignore_index=True)

    # Keep only the columns relevant for our Data Warehouse
    # (time, temp, prcp - precipitation/rain, wspd - wind speed, coco - weather condition code)
    columns_to_keep = ['airport_code', 'time', 'temp', 'prcp', 'wspd', 'coco']
    weather_df = weather_df[columns_to_keep]

    # Rename columns for clarity in our SQL database
    weather_df.rename(columns={
        'time': 'weather_timestamp',
        'temp': 'temperature_c',
        'prcp': 'precipitation_mm',
        'wspd': 'wind_speed_kmh',
        'coco': 'condition_code'
    }, inplace=True)

    # Fill any missing weather readings (e.g., missing rain data -> 0 mm)
    weather_df['precipitation_mm'] = weather_df['precipitation_mm'].fillna(0)

    print(f"✅ Weather dataframe created! Total rows: {len(weather_df)}")
    print(weather_df[['airport_code', 'weather_timestamp', 'temperature_c', 'wind_speed_kmh']].head().to_string())

    print("\nSaving cleaned datasets to CSV format...")
    # Assuming 'landed_flights' variable still exists from Cell 3 in memory
    if 'landed_flights' in locals():
        landed_flights.to_csv('clean_flights.csv', index=False, encoding='utf-8')
    weather_df.to_csv('clean_weather.csv', index=False, encoding='utf-8')

    print("✅ Extraction Complete! CSV files are updated and ready for SQL.")
else:
    print("❌ Failed to create weather dataframe.")


Starting historical weather data extraction...

Fetching hourly weather data for EPWA...
Fetching hourly weather data for EGLL...
Fetching hourly weather data for EDDF...


✅ Weather dataframe created! Total rows: 72
  airport_code   weather_timestamp  temperature_c  wind_speed_kmh
0         EPWA 2026-06-24 15:00:00           27.0            10.0
1         EPWA 2026-06-24 16:00:00           26.0            11.0
2         EPWA 2026-06-24 17:00:00           27.0            10.0
3         EPWA 2026-06-24 18:00:00           25.5            14.0
4         EPWA 2026-06-24 19:00:00           24.0             6.0

Saving cleaned datasets to CSV format...
✅ Extraction Complete! CSV files are updated and ready for SQL.
